<a href="https://colab.research.google.com/github/Chinta-Rohith/Generative-AI-Lab/blob/main/EXP9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install -U autogen-agentchat autogen-core transformers accelerate sentencepiece


In [ ]:

# ================================================================
# CUSTOMER SUPPORT AI AGENT USING AUTOGEN
# SHORT-TERM + LONG-TERM MEMORY
# HUGGING FACE QWEN MODEL - NO API KEYS
# GOOGLE COLAB
# ================================================================

import os
import json
import asyncio
import torch
from datetime import datetime
from typing import Sequence, Optional, Any

from transformers import AutoTokenizer, AutoModelForCausalLM

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import (
    ChatCompletionClient,
    CreateResult,
    RequestUsage,
    ModelInfo
)
from autogen_core import CancellationToken
from autogen_core.models import LLMMessage
from autogen_core.models import SystemMessage, UserMessage, AssistantMessage

from IPython.display import display, Markdown


# ================================================================
# STEP 1: LOAD LOCAL HUGGING FACE MODEL
# ================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Qwen model...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

model.eval()

print("Qwen model loaded successfully!")


# ================================================================
# STEP 2: CUSTOM AUTOGEN MODEL CLIENT
# Connect AutoGen to locally running Qwen
# ================================================================

class LocalQwenClient(ChatCompletionClient):

    def __init__(self, max_tokens=350):
        self.max_tokens = max_tokens

    @property
    def model_info(self) -> ModelInfo:
        return ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False
        )

    def _messages_to_prompt(self, messages: Sequence[LLMMessage]):

        converted = []

        for msg in messages:

            if isinstance(msg, SystemMessage):
                converted.append({
                    "role": "system",
                    "content": msg.content
                })

            elif isinstance(msg, UserMessage):
                converted.append({
                    "role": "user",
                    "content": str(msg.content)
                })

            elif isinstance(msg, AssistantMessage):
                converted.append({
                    "role": "assistant",
                    "content": str(msg.content)
                })

        return tokenizer.apply_chat_template(
            converted,
            tokenize=False,
            add_generation_prompt=True
        )

    async def create(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools: Sequence[Any] = [],
        json_output: Optional[bool] = None,
        extra_create_args: dict = {},
        cancellation_token: Optional[CancellationToken] = None
    ) -> CreateResult:

        prompt = self._messages_to_prompt(messages)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2500
        ).to(model.device)

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_tokens,
                do_sample=False,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        new_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        usage = RequestUsage(
            prompt_tokens=inputs["input_ids"].shape[1],
            completion_tokens=len(new_tokens)
        )

        return CreateResult(
            finish_reason="stop",
            content=response,
            usage=usage,
            cached=False
        )

    async def create_stream(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools: Sequence[Any] = [],
        json_output: Optional[bool] = None,
        extra_create_args: dict = {},
        cancellation_token: Optional[CancellationToken] = None
    ):
        result = await self.create(
            messages,
            tools=tools,
            json_output=json_output,
            extra_create_args=extra_create_args,
            cancellation_token=cancellation_token
        )

        yield result

    def count_tokens(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools: Sequence[Any] = []
    ) -> int:

        prompt = self._messages_to_prompt(messages)

        return len(
            tokenizer.encode(prompt)
        )

    def remaining_tokens(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools: Sequence[Any] = []
    ) -> int:

        return max(
            0,
            4096 - self.count_tokens(messages, tools=tools)
        )

    def actual_usage(self) -> RequestUsage:
        return RequestUsage(
            prompt_tokens=0,
            completion_tokens=0
        )

    def total_usage(self) -> RequestUsage:
        return RequestUsage(
            prompt_tokens=0,
            completion_tokens=0
        )

    async def close(self):
        pass


# Create AutoGen local model client
local_model_client = LocalQwenClient(
    max_tokens=350
)

print("AutoGen connected to local Qwen model!")


# ================================================================
# STEP 3: SHORT-TERM MEMORY
# Stores the current conversation in a list
# ================================================================

short_term_memory = []

MAX_SHORT_TERM_MESSAGES = 6


def add_short_term_memory(role, message):

    short_term_memory.append({
        "role": role,
        "message": message
    })

    # Keep only the most recent messages
    if len(short_term_memory) > MAX_SHORT_TERM_MESSAGES:

        short_term_memory.pop(0)


def get_short_term_memory():

    if not short_term_memory:
        return "No recent conversation."

    memory_text = ""

    for item in short_term_memory:

        memory_text += (
            f"{item['role']}: {item['message']}\n"
        )

    return memory_text


# ================================================================
# STEP 4: LONG-TERM MEMORY
# Stores customer history in a persistent JSON file
# ================================================================

MEMORY_FILE = "customer_long_term_memory.json"


def load_long_term_memory():

    if os.path.exists(MEMORY_FILE):

        with open(
            MEMORY_FILE,
            "r",
            encoding="utf-8"
        ) as file:

            return json.load(file)

    return {}


def save_long_term_memory(memory):

    with open(
        MEMORY_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            memory,
            file,
            indent=4,
            ensure_ascii=False
        )


long_term_memory = load_long_term_memory()


def get_customer_memory(customer_id):

    return long_term_memory.get(
        customer_id,
        {
            "name": "Unknown",
            "history": [],
            "complaints": [],
            "preferences": []
        }
    )


def update_customer_memory(
    customer_id,
    customer_name,
    user_message,
    assistant_response
):

    if customer_id not in long_term_memory:

        long_term_memory[customer_id] = {
            "name": customer_name,
            "history": [],
            "complaints": [],
            "preferences": []
        }

    customer = long_term_memory[customer_id]

    customer["name"] = customer_name

    # Save conversation history
    customer["history"].append({
        "date": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        "user": user_message,
        "assistant": assistant_response
    })

    # Simple complaint detection
    complaint_keywords = [
        "complaint",
        "problem",
        "issue",
        "not working",
        "damaged",
        "late delivery",
        "refund",
        "bad service",
        "wrong order"
    ]

    if any(
        word in user_message.lower()
        for word in complaint_keywords
    ):

        customer["complaints"].append({
            "date": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "complaint": user_message,
            "status": "Pending review"
        })

    # Keep recent history bounded
    customer["history"] = customer["history"][-20:]

    customer["complaints"] = customer["complaints"][-10:]

    save_long_term_memory(long_term_memory)


# ================================================================
# STEP 5: CREATE AUTOGEN CUSTOMER SUPPORT AGENT
# ================================================================

support_agent = AssistantAgent(

    name="CustomerSupportAgent",

    model_client=local_model_client,

    system_message="""
You are a friendly and professional AI customer support agent.

Your responsibilities:

1. Answer customer questions clearly.
2. Use short-term memory to understand the current conversation.
3. Use long-term memory to remember customer history.
4. Remember previous complaints and unresolved issues.
5. Provide personalized and empathetic assistance.
6. Avoid asking customers to repeat information already available.
7. If a complaint is unresolved, acknowledge it.
8. Never claim a refund or action is completed unless confirmed.
9. If information is unavailable, politely ask for clarification.
10. Keep responses concise and helpful.

You are working in a customer support memory management system.
"""
)

print("Customer Support Agent created successfully!")


# ================================================================
# STEP 6: CUSTOMER SUPPORT CHAT FUNCTION
# ================================================================

async def customer_support_chat():

    global short_term_memory

    print("\n==============================================")
    print(" CUSTOMER SUPPORT AI SYSTEM")
    print("==============================================")

    customer_id = input(
        "Enter Customer ID: "
    ).strip()

    if not customer_id:

        customer_id = "CUSTOMER_001"

    customer_name = input(
        "Enter Customer Name: "
    ).strip()

    if not customer_name:

        customer_name = "Customer"

    print(
        f"\nWelcome {customer_name}!"
    )

    print(
        "Type 'exit' to end the conversation."
    )

    print(
        "Type 'memory' to view saved customer memory."
    )

    print("----------------------------------------------")


    # Retrieve existing long-term memory
    customer_memory = get_customer_memory(
        customer_id
    )


    # Reset short-term memory for a new session
    short_term_memory = []


    while True:

        user_message = input(
            "\nYou: "
        ).strip()


        if user_message.lower() == "exit":

            print(
                "\nThank you for contacting customer support!"
            )

            break


        # Display long-term memory
        if user_message.lower() == "memory":

            customer_memory = get_customer_memory(
                customer_id
            )

            print("\n--- LONG-TERM CUSTOMER MEMORY ---")

            print(
                json.dumps(
                    customer_memory,
                    indent=4,
                    ensure_ascii=False
                )
            )

            continue


        # Save user input to short-term memory
        add_short_term_memory(
            "Customer",
            user_message
        )


        # Retrieve memory context
        short_memory_text = get_short_term_memory()

        customer_memory = get_customer_memory(
            customer_id
        )


        long_memory_text = json.dumps(
            customer_memory,
            indent=2,
            ensure_ascii=False
        )


        # Build personalized prompt
        prompt = f"""
CUSTOMER DETAILS

Customer ID: {customer_id}
Customer Name: {customer_name}


SHORT-TERM MEMORY
Recent conversation:
{short_memory_text}


LONG-TERM MEMORY
Customer history, preferences, and complaints:
{long_memory_text}


CURRENT CUSTOMER MESSAGE:
{user_message}


INSTRUCTIONS:

Use both short-term and long-term memory to respond.

If the customer has previous complaints, acknowledge them
and try to provide personalized assistance.

Do not invent order details, refunds, or completed actions.

Respond in a friendly, professional, empathetic manner.
"""


        # Execute AutoGen agent
        response = await support_agent.run(
            task=prompt
        )


        # Extract agent response
        assistant_response = ""

        for message in response.messages:

            if hasattr(message, "content"):

                if isinstance(
                    message.content,
                    str
                ):

                    assistant_response = message.content


        if not assistant_response:

            assistant_response = (
                "I apologize, but I could not generate "
                "a response. Please try again."
            )


        print(
            f"\nAI Support: {assistant_response}"
        )


        # Update short-term memory
        add_short_term_memory(
            "AI Support",
            assistant_response
        )


        # Update persistent long-term memory
        update_customer_memory(
            customer_id,
            customer_name,
            user_message,
            assistant_response
        )


        print(
            "\n[Short-term memory updated]"
        )

        print(
            "[Long-term customer memory saved]"
        )


# ================================================================
# STEP 7: START CUSTOMER SUPPORT SYSTEM
# ================================================================

await customer_support_chat()


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Loading Qwen model...
Device: cpu


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!


TypeError: Can't instantiate abstract class LocalQwenClient without an implementation for abstract method 'capabilities'

In [ ]:

# ================================================================
# CUSTOMER SUPPORT AI AGENT USING MICROSOFT AUTOGEN
# SHORT-TERM + LONG-TERM MEMORY
# HUGGING FACE QWEN MODEL
# NO API KEYS / NO TOKENS REQUIRED
# GOOGLE COLAB - COMPLETE CODE
# ================================================================

# STEP 1: IMPORT LIBRARIES
# ================================================================

import os
import json
import torch
import asyncio
from datetime import datetime
from typing import Sequence, Optional, Any

from transformers import AutoTokenizer, AutoModelForCausalLM

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import (
    ChatCompletionClient,
    CreateResult,
    RequestUsage,
    ModelInfo,
    ModelCapabilities,
    SystemMessage,
    UserMessage,
    AssistantMessage,
    LLMMessage
)
from autogen_core import CancellationToken

print("Libraries imported successfully!")


# ================================================================
# STEP 2: LOAD LOCAL HUGGING FACE MODEL
# ================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nLoading Qwen model...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

model.eval()

print("Qwen model loaded successfully!")


# ================================================================
# STEP 3: CUSTOM AUTOGEN MODEL CLIENT
# ================================================================

class LocalQwenClient(ChatCompletionClient):

    def __init__(self, max_tokens=300):

        self.max_tokens = max_tokens

        self._model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False
        )

        self._usage = RequestUsage(
            prompt_tokens=0,
            completion_tokens=0
        )


    # FIX: Required abstract property in newer AutoGen
    @property
    def capabilities(self) -> ModelCapabilities:

        return ModelCapabilities(
            vision=False,
            function_calling=False,
            json_output=False
        )


    @property
    def model_info(self) -> ModelInfo:

        return self._model_info


    # Convert AutoGen messages into Qwen chat format
    def messages_to_prompt(
        self,
        messages: Sequence[LLMMessage]
    ):

        converted = []

        for msg in messages:

            if isinstance(msg, SystemMessage):

                converted.append({
                    "role": "system",
                    "content": str(msg.content)
                })

            elif isinstance(msg, UserMessage):

                converted.append({
                    "role": "user",
                    "content": str(msg.content)
                })

            elif isinstance(msg, AssistantMessage):

                if isinstance(msg.content, str):

                    converted.append({
                        "role": "assistant",
                        "content": msg.content
                    })


        prompt = tokenizer.apply_chat_template(
            converted,
            tokenize=False,
            add_generation_prompt=True
        )

        return prompt


    # Generate model response
    async def create(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools=(),
        tool_choice="auto",
        json_output=None,
        extra_create_args=None,
        cancellation_token=None
    ) -> CreateResult:

        prompt = self.messages_to_prompt(messages)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2000
        ).to(model.device)

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_tokens,
                do_sample=False,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        new_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        usage = RequestUsage(
            prompt_tokens=inputs["input_ids"].shape[1],
            completion_tokens=len(new_tokens)
        )

        self._usage = usage

        return CreateResult(
            finish_reason="stop",
            content=response,
            usage=usage,
            cached=False
        )


    # Streaming support
    async def create_stream(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools=(),
        tool_choice="auto",
        json_output=None,
        extra_create_args=None,
        cancellation_token=None
    ):

        result = await self.create(
            messages,
            tools=tools,
            json_output=json_output,
            cancellation_token=cancellation_token
        )

        yield result


    # Token counting
    def count_tokens(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools=()
    ):

        prompt = self.messages_to_prompt(messages)

        return len(
            tokenizer.encode(prompt)
        )


    def remaining_tokens(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools=()
    ):

        return max(
            0,
            4096 - self.count_tokens(messages)
        )


    def actual_usage(self):

        return self._usage


    def total_usage(self):

        return self._usage


    async def close(self):

        pass


# Initialize AutoGen model client
local_model_client = LocalQwenClient(
    max_tokens=300
)

print("AutoGen connected to local Qwen model!")


# ================================================================
# STEP 4: SHORT-TERM MEMORY
# Current conversation memory
# ================================================================

short_term_memory = []

MAX_SHORT_TERM_MESSAGES = 6


def add_short_term_memory(role, message):

    short_term_memory.append({
        "role": role,
        "message": message
    })

    # Keep recent messages only
    while len(short_term_memory) > MAX_SHORT_TERM_MESSAGES:

        short_term_memory.pop(0)


def get_short_term_memory():

    if not short_term_memory:

        return "No previous messages."

    return "\n".join(
        f"{item['role']}: {item['message']}"
        for item in short_term_memory
    )


# ================================================================
# STEP 5: LONG-TERM MEMORY
# Persistent customer history in JSON
# ================================================================

MEMORY_FILE = "customer_long_term_memory.json"


def load_long_term_memory():

    if os.path.exists(MEMORY_FILE):

        try:

            with open(
                MEMORY_FILE,
                "r",
                encoding="utf-8"
            ) as file:

                return json.load(file)

        except Exception:

            return {}

    return {}


def save_long_term_memory(memory):

    with open(
        MEMORY_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            memory,
            file,
            indent=4,
            ensure_ascii=False
        )


long_term_memory = load_long_term_memory()


def get_customer_memory(customer_id):

    return long_term_memory.get(
        customer_id,
        {
            "name": "Unknown",
            "history": [],
            "complaints": [],
            "preferences": []
        }
    )


# ================================================================
# STEP 6: UPDATE CUSTOMER LONG-TERM MEMORY
# ================================================================

def update_customer_memory(
    customer_id,
    customer_name,
    user_message,
    assistant_response
):

    if customer_id not in long_term_memory:

        long_term_memory[customer_id] = {
            "name": customer_name,
            "history": [],
            "complaints": [],
            "preferences": []
        }

    customer = long_term_memory[customer_id]

    customer["name"] = customer_name


    # Save conversation history
    customer["history"].append({

        "date": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

        "user": user_message,

        "assistant": assistant_response

    })


    # Detect customer complaints
    complaint_keywords = [
        "complaint",
        "problem",
        "issue",
        "not working",
        "damaged",
        "late delivery",
        "refund",
        "bad service",
        "wrong order",
        "defective"
    ]


    if any(
        word in user_message.lower()
        for word in complaint_keywords
    ):

        customer["complaints"].append({

            "date": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            "complaint": user_message,

            "status": "Pending review"

        })


    # Keep memory within manageable size
    customer["history"] = customer["history"][-20:]

    customer["complaints"] = customer["complaints"][-10:]


    # Save memory permanently to JSON file
    save_long_term_memory(long_term_memory)


# ================================================================
# STEP 7: CREATE AUTOGEN CUSTOMER SUPPORT AGENT
# ================================================================

support_agent = AssistantAgent(

    name="CustomerSupportAgent",

    model_client=local_model_client,

    system_message="""

You are a friendly and professional AI customer support agent.

Your responsibilities:

1. Answer customer questions clearly.
2. Use short-term memory to understand the current conversation.
3. Use long-term memory to remember customer history.
4. Remember previous complaints and unresolved issues.
5. Provide personalized and empathetic assistance.
6. Avoid asking customers to repeat information already known.
7. Acknowledge unresolved complaints.
8. Never claim a refund or action is completed without confirmation.
9. Ask for clarification when necessary.
10. Keep responses concise, helpful, and professional.

You are part of an AI customer support memory system.

"""

)

print("Customer Support Agent created successfully!")


# ================================================================
# STEP 8: CUSTOMER SUPPORT CHAT FUNCTION
# ================================================================

async def customer_support_chat():

    global short_term_memory

    print("\n============================================")
    print("      CUSTOMER SUPPORT AI SYSTEM")
    print("============================================")

    customer_id = input(
        "\nEnter Customer ID: "
    ).strip()

    if not customer_id:

        customer_id = "CUSTOMER_001"


    customer_name = input(
        "Enter Customer Name: "
    ).strip()

    if not customer_name:

        customer_name = "Customer"


    print(f"\nWelcome, {customer_name}!")

    print("\nCommands:")
    print("1. Type your question to chat")
    print("2. Type 'memory' to view saved history")
    print("3. Type 'exit' to end chat")

    print("--------------------------------------------")


    # Retrieve customer history
    customer_memory = get_customer_memory(
        customer_id
    )


    # Reset short-term memory for new session
    short_term_memory = []


    while True:

        user_message = input(
            "\nYou: "
        ).strip()


        # Exit
        if user_message.lower() == "exit":

            print(
                "\nThank you for contacting customer support!"
            )

            print(
                "Your conversation history has been saved."
            )

            break


        # Display long-term memory
        if user_message.lower() == "memory":

            customer_memory = get_customer_memory(
                customer_id
            )

            print(
                "\n--- LONG-TERM CUSTOMER MEMORY ---"
            )

            print(
                json.dumps(
                    customer_memory,
                    indent=4,
                    ensure_ascii=False
                )
            )

            continue


        # Store user message in short-term memory
        add_short_term_memory(
            "Customer",
            user_message
        )


        # Retrieve both memory types
        short_memory_text = get_short_term_memory()

        customer_memory = get_customer_memory(
            customer_id
        )

        long_memory_text = json.dumps(
            customer_memory,
            indent=2,
            ensure_ascii=False
        )


        # Create context-aware prompt
        prompt = f"""

CUSTOMER INFORMATION

Customer ID: {customer_id}

Customer Name: {customer_name}


SHORT-TERM MEMORY
Current conversation:
{short_memory_text}


LONG-TERM MEMORY
Previous customer history and complaints:
{long_memory_text}


CURRENT CUSTOMER MESSAGE:
{user_message}


INSTRUCTIONS:

Use short-term and long-term memory to understand the customer.

If the customer has a previous complaint, acknowledge it.

Provide personalized, friendly, and empathetic assistance.

Do not invent order information, refunds, or completed actions.

Respond clearly and concisely.

"""


        # Run AutoGen agent
        response = await support_agent.run(
            task=prompt
        )


        # Extract response
        assistant_response = ""

        for message in response.messages:

            if hasattr(message, "content"):

                if isinstance(
                    message.content,
                    str
                ):

                    assistant_response = message.content


        if not assistant_response:

            assistant_response = (
                "Sorry, I could not generate a response. "
                "Please try again."
            )


        # Display response
        print(
            f"\nAI Support: {assistant_response}"
        )


        # Update short-term memory
        add_short_term_memory(
            "AI Support",
            assistant_response
        )


        # Update long-term memory
        update_customer_memory(
            customer_id,
            customer_name,
            user_message,
            assistant_response
        )


        print(
            "\n[Short-term memory updated]"
        )

        print(
            "[Long-term memory saved successfully]"
        )


# ================================================================
# STEP 9: START CUSTOMER SUPPORT CHAT
# ================================================================

await customer_support_chat()


Libraries imported successfully!

Loading Qwen model...
Device: cpu


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model loaded successfully!
AutoGen connected to local Qwen model!
Customer Support Agent created successfully!

      CUSTOMER SUPPORT AI SYSTEM

Enter Customer ID: C101
Enter Customer Name: Rahul

Welcome, Rahul!

Commands:
1. Type your question to chat
2. Type 'memory' to view saved history
3. Type 'exit' to end chat
--------------------------------------------

You: My order arrived damaged. I want a replacement.

AI Support: Hello Rahul! Thank you for reaching out with your concern about the damage on your order. We're sorry to hear that you've experienced this issue. Could you please provide more details so we can assist you better? Specifically, what items were in your order, and how was the package delivered? This will help us identify any potential causes and ensure a prompt resolution.

[Short-term memory updated]
[Long-term memory saved successfully]

You: What did I just tell you about my order?

AI Support: I'm sorry, but I didn't quite catch that. When you said "What 